# 02.1 Image Data and Transforms

Computer vision code becomes much easier once image shape is clear. In PyTorch, an image tensor usually stores channels first, so one grayscale image has shape `(1, H, W)` and a batch has shape `(N, C, H, W)`. Many external libraries display images with channels last, so part of this notebook is learning when shapes need to be rearranged.

The second idea is preprocessing. Raw image values are often scaled, normalized, and sometimes augmented before entering a model. These steps are not cosmetic; they change the numeric input distribution the model sees.

## Learning Goals

After this notebook, you should be able to:

1. Understand the standard shape of image tensors.
2. Distinguish a single image from a batch of images.
3. Use `torchvision.transforms` to build a basic preprocessing pipeline.
4. Understand why scaling and normalization are common.
5. Make a custom dataset support transforms.
6. Prepare data input for later CNN classification tasks.

In [ ]:
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import load_digits
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


## What Shape Does One Image Have in PyTorch?

The common PyTorch convention is channel-first. One grayscale image is `(C, H, W)` with `C=1`, and one RGB image is `(C, H, W)` with `C=3`. A batch adds a leading sample dimension, so the batch shape is `(N, C, H, W)`.

This differs from many plotting and image-processing tools, which often use `(H, W, C)`. When a plot looks wrong or a convolution layer rejects an input, check whether the channel dimension is in the expected position.

In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

print("images.shape =", images.shape)
print("labels.shape =", labels.shape)
print("one raw image shape / one raw image shape =", images[0].shape)

A single image from `sklearn digits` has shape `(8, 8)`, meaning it only has height and width.

Because it is grayscale, we usually explicitly add the channel dimension.


In [ ]:
raw_img = torch.tensor(images[0], dtype=torch.float32)
img_chw = raw_img.unsqueeze(0)

print("raw_img.shape =", raw_img.shape)
print("img_chw.shape =", img_chw.shape)

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(raw_img, cmap="gray")
plt.title(f"Digit / digit: {labels[0]}")
plt.axis("off")
plt.show()

## Single Images vs Batches of Images

A single image has no batch dimension. A model training step normally receives many images at once, so DataLoader stacks individual images into a batch. That is why `(C, H, W)` becomes `(N, C, H, W)`.

The model processes all `N` images in parallel, while the channel, height, and width dimensions describe the structure of each image.

In [ ]:
batch = torch.stack([
    torch.tensor(images[0], dtype=torch.float32).unsqueeze(0),
    torch.tensor(images[1], dtype=torch.float32).unsqueeze(0),
    torch.tensor(images[2], dtype=torch.float32).unsqueeze(0),
])

print("batch.shape =", batch.shape)

`batch.shape == (3, 1, 8, 8)` means there are 3 images in the batch. Each image has 1 channel because the digits are grayscale. The final two dimensions are height 8 and width 8. Read the shape as `(batch, channels, height, width)`.

In [ ]:
# Exercise 1
#
# Turn the first 5 images in the digits dataset into one batch.
#
# Steps:
# - For each of the first 5 images, create a float32 tensor.
# - Add a channel dimension so each image has shape (1, 8, 8).
# - Stack the 5 image tensors into one batch.
#
# Expected final shape:
# - batch5.shape should be (5, 1, 8, 8).

# imgs =
# batch5 =
# print(batch5.shape)

In [ ]:
# Exercise 1 Reference Solution

imgs = [torch.tensor(images[i], dtype=torch.float32).unsqueeze(0) for i in range(5)]
batch5 = torch.stack(imgs)
print(batch5.shape)

## Why Use Transforms?

Transforms define the preprocessing that happens every time a sample is read. Scaling pixel values into a small numeric range makes optimization easier. Normalization moves values into a distribution the model can handle more consistently. Augmentation creates modified training examples so the model does not rely too heavily on one exact view of an image.

The important design point is that preprocessing should live in the dataset or transform pipeline, not as scattered manual edits across the notebook.

In [ ]:
print("raw pixel range / raw pixel range:", raw_img.min().item(), raw_img.max().item())

The pixel values in the `digits` dataset roughly range from `0` to `16`.

normalization.  
A common practice is to first scale them into `0~1`, and then normalize them.

In [ ]:
preprocess = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

img_processed = preprocess(img_chw)

print("before preprocessing / before preprocessing:", img_chw.min().item(), img_chw.max().item())
print("after preprocessing / after preprocessing:", img_processed.min().item(), img_processed.max().item())
print("processed mean / processed mean:", img_processed.mean().item())

`Normalize(mean=[0.5], std=[0.5])` means that each pixel is transformed by `(pixel - 0.5) / 0.5`. If the input pixels were already scaled to `0` through `1`, this maps the rough range to `-1` through `1`. That gives the model inputs centered around zero, which is often easier to optimize than raw positive pixel values.

## Making a Custom Dataset Support Transforms

Real image projects usually define a dataset class with a `transform` argument. The dataset is responsible for reading one image and label. The transform is responsible for turning the raw image into the tensor format and numeric scale expected by the model.

Applying the transform inside `__getitem__` is important because it ensures every sample goes through the same preprocessing path when DataLoader requests it.

In [ ]:
class DigitsImageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.labels[index], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        return image, label


dataset = DigitsImageDataset(images, labels, transform=preprocess)
img0, label0 = dataset[0]
print("img0.shape =", img0.shape)
print("label0 =", label0)
print("img0.dtype =", img0.dtype)

In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=False)
xb, yb = next(iter(loader))

print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)
print("xb.min() =", xb.min().item())
print("xb.max() =", xb.max().item())

In [ ]:
# Exercise 2
#
# Implement a simple transform that scales pixel values to 0 through 1.
#
# Requirements:
# - Do not use Normalize in this exercise.
# - Use a transform that converts an image tensor to float and divides by 16.0,
#   because the sklearn digits pixel range is 0 through 16.
# - Create simple_dataset with that transform.
# - Read simple_dataset[0] and print the image min and max values.

# simple_transform =
# simple_dataset =
# img, label =
# print(img.min().item(), img.max().item())

In [ ]:
# Exercise 2 Reference Solution

simple_transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
])
simple_dataset = DigitsImageDataset(images, labels, transform=simple_transform)
img, label = simple_dataset[0]
print(img.min().item(), img.max().item())

## About Data Augmentation

Data augmentation creates modified versions of training images, such as random crops or rotations. It can improve generalization when the modification preserves the label. It can also hurt when the modification changes the meaning of the image.

For digit recognition, horizontal flipping is a good example of a risky augmentation because a flipped digit may no longer represent the same class. Always ask whether the transformed image still has the original label before adding an augmentation.

## Summary

The most important goal of this notebook is to make image data format completely clear.

You should now be able to answer:

1. Why does `PyTorch` usually use `(C, H, W)` instead of `(H, W, C)`?
2. What is the difference between the shape of one image and a batch of images?
3. Why are images often scaled and then normalized?
4. Why is `transform` often built into the `Dataset`?

Suggested next step:

- Move to the CNN basics notebook and understand what convolution layers do to images.